# Conversión `.extxyz` → `.xyz` unificado + dataset combinado + train/test + PCA / t-SNE / UMAP

Cada fichero de salida tiene solo:

```
Lattice="..." Properties=species:S:1:pos:R:3:forces_qe:R:3 energy_qe=... config_type=... pbc="T T T"
```

- Detecta automáticamente cómo se llaman la energía y las fuerzas en cada fichero (`energy_qe`, `REF_energy`, `energy`, `forces`, `REF_forces`, ...) y las renombra a `energy_qe` / `forces_qe`.
- Los números se copian tal cual (sin redondear).
- Para añadir ficheros, solo hay que escribir su nombre en `FILES`.
- Al final, compara los subconjuntos (un color por fichero) con PCA, t-SNE y UMAP: primero el total y después train y test.

In [ ]:
# ============================ CONFIGURACIÓN ============================
FILES = [
    "strain.extxyz",
    "rattle.extxyz",
    "aimd.extxyz",
    "aimd_defective.extxyz",
    "postrotura_scf.extxyz",
    "vacancies.xyz",      # <- añade aquí todos los que quieras
]

OUT_DIR       = "dataset_xyz_v33"      # carpeta de salida
COMBINED_NAME = "dataset_all_v33.xyz"
TRAIN_NAME    = "train_v33.xyz"
TEST_NAME     = "test_v33.xyz"

ENERGY_OUT = "energy_qe"           # nombre de la energía en la salida
FORCES_OUT = "forces_qe"           # nombre de las fuerzas en la salida
PBC_OUT    = "T T T"               # None -> conserva el pbc original de cada fichero

# Nombres que se buscan (en este orden) en los ficheros de entrada
ENERGY_KEYS = ["energy_qe", "REF_energy", "energy", "dft_energy", "DFT_energy"]
FORCES_KEYS = ["forces_qe", "REF_forces", "forces", "dft_forces", "DFT_forces"]

PREFIX_CONFIG_TYPE = False         # True -> config_type = <fichero>_<config_type>  (p. ej. aimd_T500K)

# --- División train / test ---
DO_SPLIT     = True
TEST_PERCENT = 10                  # % de configuraciones que van al test set
SEED         = 42                  # semilla del reparto aleatorio
STRATIFY     = True                # True -> el mismo % de test dentro de cada config_type

# --- Visualización PCA / t-SNE / UMAP ---
DO_VIS     = True
DESCRIPTOR = "soap"                # "soap" (requiere dscribe) o "rdf" (solo numpy + ASE)
R_CUT      = 5.0                   # radio de corte del descriptor (Å)
PERPLEXITY = 30                    # t-SNE
N_NEIGHBORS, MIN_DIST = 15, 0.1    # UMAP

In [ ]:
import re, random
from pathlib import Path
from collections import Counter, defaultdict

KV_RE = re.compile(r'([^\s=]+)=("[^"]*"|\S+)')

def parse_header(line):
    """Línea de comentario extxyz -> dict {clave: valor} (sin comillas)."""
    return {k: v.strip('"') for k, v in KV_RE.findall(line)}

def column_slices(properties):
    """'species:S:1:pos:R:3:forces:R:3' -> {'species': (0,1), 'pos': (1,4), 'forces': (4,7)}"""
    t, cols, i = properties.split(":"), {}, 0
    for name, n in zip(t[0::3], t[2::3]):
        cols[name] = (i, i + int(n))
        i += int(n)
    return cols

def first_key(d, keys, what, fname):
    for k in keys:
        if k in d:
            return k
    raise KeyError(f"{fname}: no encuentro {what}. Disponibles: {list(d)}")

def convert_file(path):
    """Lee un .extxyz y devuelve [(config_type, texto_del_frame), ...] en el formato unificado."""
    path = Path(path)
    lines = path.read_text().splitlines()
    frames, used, i = [], Counter(), 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        nat  = int(lines[i])
        h    = parse_header(lines[i + 1])
        cols = column_slices(h["Properties"])
        ek   = first_key(h, ENERGY_KEYS, "la energía", path.name)
        fk   = first_key(cols, FORCES_KEYS, "las fuerzas", path.name)
        used[(ek, fk)] += 1

        ct = h.get("config_type", path.stem)
        if PREFIX_CONFIG_TYPE:
            ct = f"{path.stem}_{ct}"
        ct_str = f'"{ct}"' if " " in ct else ct
        pbc = PBC_OUT or h.get("pbc", "T T T")

        header = (f'Lattice="{h["Lattice"]}" '
                  f'Properties=species:S:1:pos:R:3:{FORCES_OUT}:R:3 '
                  f'{ENERGY_OUT}={h[ek]} config_type={ct_str} pbc="{pbc}"')

        (s0, _), (p0, p1), (f0, f1) = cols["species"], cols["pos"], cols[fk]
        atoms = []
        for l in lines[i + 2 : i + 2 + nat]:
            t = l.split()
            atoms.append(f"{t[s0]:<3}" + "".join(f"{x:>16}" for x in t[p0:p1] + t[f0:f1]))

        frames.append((ct, "\n".join([str(nat), header, *atoms])))
        i += nat + 2
    return frames, used

def write_xyz(frames, fname):
    with open(fname, "w") as f:
        f.write("\n".join(txt for _, txt in frames) + "\n")

def print_table(groups):
    """groups: {nombre_columna: lista_de_frames}"""
    counts = {k: Counter(ct for ct, _ in v) for k, v in groups.items()}
    cts = list(dict.fromkeys(ct for v in groups.values() for ct, _ in v))
    print(f"{'config_type':<22}" + "".join(f"{k:>9}" for k in groups))
    for ct in cts:
        print(f"{ct:<22}" + "".join(f"{counts[k][ct]:>9}" for k in groups))
    print(f"{'TOTAL':<22}" + "".join(f"{len(v):>9}" for v in groups.values()))

## 1. Convertir cada fichero

In [ ]:
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

data = {}
for f in FILES:
    frames, used = convert_file(f)
    data[f] = frames
    write_xyz(frames, out / f"{Path(f).stem}.xyz")
    keys = ", ".join(f"{e}/{fo}" for e, fo in used)
    print(f"{f:<22} {len(frames):>6} configs  (energía/fuerzas leídas: {keys})  ->  {out / (Path(f).stem + '.xyz')}")

## 2. Dataset combinado

In [ ]:
all_frames = [fr for f in FILES for fr in data[f]]
write_xyz(all_frames, out / COMBINED_NAME)
print(f"Guardado {out / COMBINED_NAME}\n")
print_table({"total": all_frames})

## 3. División train / test (aleatoria con semilla)

In [ ]:
def split(frames, test_percent, seed, stratify=True):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for idx, (ct, _) in enumerate(frames):
        groups[ct if stratify else "all"].append(idx)
    test_idx = set()
    for idx in groups.values():
        rng.shuffle(idx)
        test_idx.update(idx[: round(len(idx) * test_percent / 100)])
    train = [fr for i, fr in enumerate(frames) if i not in test_idx]
    test  = [fr for i, fr in enumerate(frames) if i in test_idx]
    return train, test, test_idx

test_idx = set()
if DO_SPLIT:
    train, test, test_idx = split(all_frames, TEST_PERCENT, SEED, STRATIFY)
    write_xyz(train, out / TRAIN_NAME)
    write_xyz(test,  out / TEST_NAME)
    print(f"Test = {TEST_PERCENT}%  |  seed = {SEED}  |  estratificado = {STRATIFY}")
    print(f"Guardados {out / TRAIN_NAME} y {out / TEST_NAME}\n")
    print_table({"total": all_frames, "train": train, "test": test})

## 4. (Opcional) Comprobar que los ficheros se leen bien con ASE

In [ ]:
try:
    from ase.io import read
    for name in [COMBINED_NAME, TRAIN_NAME, TEST_NAME]:
        p = out / name
        if p.exists():
            images = read(p, index=":")
            a = images[0]
            print(f"{name:<18} {len(images):>6} configs | info: {list(a.info)} | arrays: {list(a.arrays)} | pbc: {a.pbc}")
except ImportError:
    print("ASE no está instalado: se omite la comprobación (pip install ase)")

## 5. Visualización comparativa: PCA, t-SNE y UMAP

- Cada configuración se describe con un vector estructural: **SOAP promediado** sobre los átomos y normalizado (con `dscribe`) o, si no está instalado, las **RDF parciales** por pareja de especies (B–B, B–C, C–C).
- Las tres proyecciones se ajustan **una sola vez sobre el total**, y train y test se dibujan sobre esas mismas coordenadas, así que son directamente comparables.
- Color y símbolo = subconjunto (un fichero de `FILES`). Las figuras se guardan en `OUT_DIR` en PNG (300 dpi) y las coordenadas en `proyecciones.csv`.

Requisitos: `pip install scikit-learn matplotlib umap-learn dscribe`

In [ ]:
import numpy as np
from itertools import combinations_with_replacement
from ase.io import read
from ase.neighborlist import neighbor_list

images  = read(out / COMBINED_NAME, index=":")              # mismo orden que all_frames
subset  = np.array([Path(f).stem for f in FILES for _ in data[f]])
is_test = np.zeros(len(images), bool)
is_test[list(test_idx)] = True
species = sorted({s for a in images for s in a.get_chemical_symbols()})

def rdf_descriptor(images, species, r_cut, nbins=120, sigma=0.08):
    """RDF parciales suavizadas (gaussianas) por pareja de especies, normalizadas por átomo."""
    r = np.linspace(0.8, r_cut, nbins)
    pairs = list(combinations_with_replacement(species, 2))
    X = np.zeros((len(images), len(pairs) * nbins))
    for n, a in enumerate(images):
        sym = np.array(a.get_chemical_symbols())
        i, j, d = neighbor_list("ijd", a, r_cut)
        for k, (s1, s2) in enumerate(pairs):
            dm = d[(sym[i] == s1) & (sym[j] == s2)]
            g = np.exp(-0.5 * ((r[None, :] - dm[:, None]) / sigma) ** 2).sum(0)
            X[n, k * nbins:(k + 1) * nbins] = g / max(1, (sym == s1).sum())
    return X

if DO_VIS:
    if DESCRIPTOR == "soap":
        try:
            from dscribe.descriptors import SOAP
        except ImportError:
            print("dscribe no está instalado -> uso RDF (pip install dscribe para SOAP)")
            DESCRIPTOR = "rdf"
    if DESCRIPTOR == "soap":
        soap = SOAP(species=species, r_cut=R_CUT, n_max=6, l_max=4,
                    periodic=True, average="inner", sparse=False)
        X = soap.create(images, n_jobs=1)
        X /= np.linalg.norm(X, axis=1, keepdims=True)           # normaliza: quita el efecto de la densidad
    else:
        X = rdf_descriptor(images, species, R_CUT)
    print(f"Descriptor {DESCRIPTOR.upper()}: {X.shape[0]} configuraciones x {X.shape[1]} componentes | especies {species}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

if DO_VIS:
    pca = PCA(n_components=2, random_state=SEED).fit(X)
    emb = {"PCA": pca.transform(X)}

    Xr = PCA(n_components=min(50, *X.shape), random_state=SEED).fit_transform(X)   # pre-reducción para t-SNE / UMAP
    emb["t-SNE"] = TSNE(n_components=2, perplexity=min(PERPLEXITY, (len(X) - 1) // 3),
                        init="pca", learning_rate="auto", random_state=SEED).fit_transform(Xr)
    try:
        import umap
        emb["UMAP"] = umap.UMAP(n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                                random_state=SEED).fit_transform(Xr)
    except ImportError:
        print("umap-learn no está instalado: se omite UMAP (pip install umap-learn)")

    print("Varianza explicada PCA:", ", ".join(f"PC{k+1} = {v:.1%}" for k, v in enumerate(pca.explained_variance_ratio_)))

In [ ]:
import csv
import matplotlib.pyplot as plt

COLORS  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

names = list(dict.fromkeys(subset))                                   # orden de FILES
style = {s: (COLORS[k % 8], MARKERS[k % 8]) for k, s in enumerate(names)}
draw_order = sorted(names, key=lambda s: -(subset == s).sum())        # el más grande debajo

def limits(E, pad=0.05):
    lo, hi = E.min(0), E.max(0)
    return np.c_[lo - pad * (hi - lo), hi + pad * (hi - lo)]

def axis_labels(ax, method):
    if method == "PCA":
        v = pca.explained_variance_ratio_
        ax.set_xlabel(f"PC1 ({v[0]:.0%})"); ax.set_ylabel(f"PC2 ({v[1]:.0%})")
    else:
        ax.set_xlabel(f"{method} 1"); ax.set_ylabel(f"{method} 2")

def panel(ax, method, mask, title, background=None, size=14):
    E = emb[method]
    if background is not None:
        ax.scatter(*E[background].T, s=8, c="#d9d9d6", linewidths=0, label=f"train ({background.sum()})")
    for s in draw_order:
        m = mask & (subset == s)
        c, mk = style[s]
        ax.scatter(*E[m].T, s=size, c=c, marker=mk, edgecolors="white", linewidths=0.4,
                   label=f"{s} ({m.sum()})")
    (x0, x1), (y0, y1) = limits(E)                                   # mismos ejes en todas las filas
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("#c3c2b7")
    axis_labels(ax, method)
    ax.set_title(title, fontsize=11)

def legend_right(ax, title):
    h, l = ax.get_legend_handles_labels()                              # leyenda en el orden de FILES
    rank = lambda lab: -1 if lab.startswith("train") else names.index(lab.rsplit(" (", 1)[0])
    h, l = zip(*sorted(zip(h, l), key=lambda hl: rank(hl[1])))
    ax.legend(h, l, title=title, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
              fontsize=9, title_fontsize=9, markerscale=1.6, handletextpad=0.3)

if DO_VIS:
    methods = list(emb)
    every = np.ones(len(subset), bool)

    # ---------- Figura 1: total ----------
    fig1, axs = plt.subplots(1, len(methods), figsize=(4.6 * len(methods) + 1.6, 4.6), squeeze=False)
    for ax, m in zip(axs[0], methods):
        panel(ax, m, every, f"{m} · total")
    legend_right(axs[0, -1], f"Total ({len(subset)})")
    fig1.suptitle(f"Dataset completo · descriptor {DESCRIPTOR.upper()}", fontsize=12)
    fig1.tight_layout()
    fig1.savefig(out / "proyecciones_total.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ---------- Figura 2: train y test ----------
    if DO_SPLIT:
        fig2, axs = plt.subplots(2, len(methods), figsize=(4.6 * len(methods) + 1.6, 9.2), squeeze=False)
        for ax, m in zip(axs[0], methods):
            panel(ax, m, ~is_test, f"{m} · train")
        for ax, m in zip(axs[1], methods):
            panel(ax, m, is_test, f"{m} · test (gris = train)", background=~is_test, size=30)
        legend_right(axs[0, -1], f"Train ({(~is_test).sum()})")
        legend_right(axs[1, -1], f"Test ({is_test.sum()})")
        fig2.suptitle(f"Train vs test ({TEST_PERCENT}% test, seed = {SEED}) · descriptor {DESCRIPTOR.upper()}", fontsize=12)
        fig2.tight_layout()
        fig2.savefig(out / "proyecciones_train_test.png", dpi=300, bbox_inches="tight")
        plt.show()

    # ---------- Coordenadas a CSV (para re-dibujar en Origin, etc.) ----------
    with open(out / "proyecciones.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["subset", "config_type", "split"] + [f"{m}_{i}" for m in methods for i in (1, 2)])
        for n in range(len(subset)):
            w.writerow([subset[n], images[n].info["config_type"], "test" if is_test[n] else "train"]
                       + [f"{emb[m][n, i]:.5f}" for m in methods for i in (0, 1)])
    print(f"Guardados en {out}: proyecciones_total.png, proyecciones_train_test.png, proyecciones.csv")